# Financial Statements in the OpenBB Platform

OpenBB Platform data extensions provide access to financial statements as quarterly or annual.  There are also endpoints for ratios and other common non-GAAP metrics.  Most data providers require a subscription to access all data. Refer to the website of a specific provider for details on entitlements and coverage.

Financial statement functions are grouped under the `obb.equity.fundamental` module.

## Endpoints

The typical financial statements consist of three endpoints:

- Balance Sheet: `obb.equity.fundamental.balance()`
- Income Statement: `obb.equity.fundamental.income()`
- Cash Flow Statement: `obb.equity.fundamental.cash()`

The main parameters are:

- `symbol`: The company's symbol.
- `period`: 'annual' or 'quarter'. Default is 'annual'.
- `limit`: Limit the number of results returned, from the latest. Default is 5. For perspective, 150 will go back to 1985. The amount of historical records varies by provider.

### Field Names

Some considerations to keep in mind when working with financial statements data are:

- Every data provider has their own way of parsing and organizing the three financial statements.
- Items within each statement will vary by source and by the type of company reporting.
- Names of line items will vary by source.
- "Date" values may differ because they are from the period starting/ending or date of reporting.

This example highlights how different providers will have different labels for compnay facts.

**Note**: API Keys are required for FMP, Intrinio, and Polygon.

In [48]:
import pandas as pd
from openbb import obb

In [49]:
df = pd.DataFrame()

df["yfinance"] = (
    obb.equity.fundamental.balance(
        "TGT", provider="yfinance"
    )  # There is no limit for yFinance, historical data is limited.
    .to_dataframe()  # updated method name
    .get("total_assets")
    .head(3)
)

df["fmp"] = (
    obb.equity.fundamental.balance("TGT", provider="fmp", limit=3)
    .to_dataframe()  # updated method name
    .get("total_assets")
)

df["intrinio"] = (
    obb.equity.fundamental.balance("TGT", provider="intrinio", limit=3)
    .to_dataframe()  # updated method name
    .get("total_assets")
)

df["polygon"] = (
    obb.equity.fundamental.balance("TGT", provider="polygon", limit=3)
    .to_dataframe()  # updated method name
    .get("total_assets")
)

df

,yfinance,fmp,intrinio,polygon
0,5.535600e+10,5.535600e+10,5.535600e+10,5.535600e+10
1,5.333500e+10,5.333500e+10,5.333500e+10,5.333500e+10
2,5.381100e+10,5.381100e+10,5.381100e+10,5.381100e+10


### Weighted Average Shares Outstanding

This key metric will be found under the income statement.  It might also be called, 'basic', and the numbers do not include authorized but unissued shares.  A declining count over time is a sign that the company is returning capital to shareholders in the form of buy backs.  Under ideal circumstances, it is more capital-efficient, for both company and shareholders, because distributions are double-taxed.  The company pays income tax on paid dividends, and the beneficiary pays income tax again on receipt.

A company will disclose how many shares are outstanding at the end of the period  as a weighted average over the reporting period - three months.

Let's take a look at Target.  To make the numbers easier to read, we'll divide the entire column by one million.

In [50]:
data = obb.equity.fundamental.income(
    "TGT", provider="fmp", limit=150, period="quarter"
).to_dataframe()  # updated method name

shares = data["weighted_average_basic_shares_outstanding"] / 1000000

display(shares.head(1))

display(shares.tail(1))

0    462.5
Name: weighted_average_basic_shares_outstanding, dtype: float64

149    1169.248
Name: weighted_average_basic_shares_outstanding, dtype: float64

Thirty-seven years later, the share count is approaching a two-thirds reduction.  12.2% over the past five years.  In four reporting periods, 1.3 million shares have been taken out of the float.

In [51]:
display(shares.pct_change(20).iloc[-1])

display(shares.iloc[-4] - shares.iloc[-1])

0.3362834285714287

-65.75199999999995

With an average closing price of $143.37, that represents approximately $190M in buy backs.

In [52]:
price = obb.equity.price.historical(
    "TGT", start_date="2022-10-01", provider="fmp"
).to_dataframe()  # updated method name

round((price["close"].mean() * 1300000) / 1000000, 2)

190.75

### Dividends Paid

Dividends paid is in the cash flow statement.  We can calculate the amount-per-share with the reported data.

In [54]:
dividends = obb.equity.fundamental.cash(
    "TGT", provider="fmp", limit=150, period="quarter"
).to_dataframe()[["payment_of_dividends"]]  # updated method name

dividends[["shares"]] = data[["weighted_average_basic_shares_outstanding"]]
dividends[["div_per_share"]] = abs(
    dividends[["payment_of_dividends"]] / dividends[["shares"]]
)

dividends[["div_per_share"]].tail(4)

136    0.040339
137    0.023793
138    0.020690
139    0.022969
Name: div_per_share, dtype: float64

This can be compared against the real amounts paid to common share holders, as announced.  Note that the dates above represent the report date, and that dividends paid are attributed to the quarter they were paid in.  The value from "2023-01-28" equates to the fourth quarter of 2022.